In [1]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

# Connessione al database
db_path = '../data/document_analysis.db'
conn = sqlite3.connect(db_path)

print("--- STATISTICHE GENERALI ---")
# Conteggio documenti e triplette
n_docs = pd.read_sql_query("SELECT COUNT(*) FROM documents", conn).iloc[0,0]
n_triples = pd.read_sql_query("SELECT COUNT(*) FROM rdf_triples", conn).iloc[0,0]
print(f"Documenti totali: {n_docs}")
print(f"Relazioni (triplette) estratte: {n_triples}")

print("\n--- TOP 10 PERSONAGGI/ENTITÀ (Per la RQ dei Broker) ---")
# Chi sono i più citati? (Sia come attori che come bersagli)
query_entities = """
SELECT entity, COUNT(*) as frequenza FROM (
    SELECT actor as entity FROM rdf_triples
    UNION ALL
    SELECT target as entity FROM rdf_triples
) GROUP BY entity ORDER BY frequenza DESC LIMIT 10
"""
top_entities = pd.read_sql_query(query_entities, conn)
print(top_entities)

print("\n--- DISTRIBUZIONE TEMPORALE (Per la RQ Evoluzione) ---")
# Vediamo se i timestamp sono utilizzabili
query_time = "SELECT timestamp FROM rdf_triples WHERE timestamp IS NOT NULL"
df_time = pd.read_sql_query(query_time, conn)
# Nota: potrebbe servire pulizia sui dati temporali qui
print(f"Relazioni con data certa: {len(df_time)}")

conn.close()

--- STATISTICHE GENERALI ---
Documenti totali: 25232
Relazioni (triplette) estratte: 107030

--- TOP 10 PERSONAGGI/ENTITÀ (Per la RQ dei Broker) ---
              entity  frequenza
0    Jeffrey Epstein      27590
1   unknown person A       8335
2       Donald Trump       3813
3    Alan Dershowitz       1860
4     Edward Snowden       1843
5         Ehud Barak       1686
6   unknown person B       1347
7         Jeffrey E.       1299
8  Ghislaine Maxwell       1289
9       Steve Bannon       1244

--- DISTRIBUZIONE TEMPORALE (Per la RQ Evoluzione) ---
Relazioni con data certa: 61481


In [2]:
import sqlite3
import pandas as pd

# 1. Connessione
db_path = '../data/document_analysis.db'
conn = sqlite3.connect(db_path)

# 2. Carichiamo la tabella degli alias (la nostra "stele di rosetta")
aliases_df = pd.read_sql_query("SELECT original_name, canonical_name FROM entity_aliases", conn)
# Creiamo un dizionario per una sostituzione velocissima: { 'Jeffrey E.': 'Jeffrey Epstein', ... }
alias_map = dict(zip(aliases_df['original_name'], aliases_df['canonical_name']))

# 3. Carichiamo le triplette
triples_df = pd.read_sql_query("SELECT actor, action, target, timestamp FROM rdf_triples", conn)

# 4. Funzione di pulizia: se il nome è nella mappa degli alias lo cambiamo, altrimenti lo teniamo
def clean_entity(name):
    return alias_map.get(name, name)

# Applichiamo la pulizia ad attori e bersagli
triples_df['actor'] = triples_df['actor'].apply(clean_entity)
triples_df['target'] = triples_df['target'].apply(clean_entity)

print(f"Deduplicazione completata su {len(triples_df)} relazioni.")

# 5. Ricalcoliamo la Top 10 PULITA
all_entities = pd.concat([triples_df['actor'], triples_df['target']])
top_10_clean = all_entities.value_counts().head(10)

print("\n--- NUOVA TOP 10 (Deduplicata) ---")
print(top_10_clean)

conn.close()

Deduplicazione completata su 107030 relazioni.

--- NUOVA TOP 10 (Deduplicata) ---
Jeffrey Epstein       29345
unknown person A       8335
Donald Trump           4082
Alan M. Dershowitz     2082
Edward Snowden         1900
Ehud Barak             1773
unknown person B       1347
Steve Bannon           1301
Ghislaine Maxwell      1294
Bill Clinton           1166
Name: count, dtype: int64


In [3]:
print(triples_df['timestamp'].dropna().unique()[:10])

['1989' '1997-04-08' '1991-11' '1995' '2004-09-08' '2005-03-15' '2005'
 '2005-05' '2008-06' '2016-01']


In [4]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Estraiamo l'anno (prendiamo i primi 4 caratteri di ogni stringa non nulla)
triples_df['year'] = triples_df['timestamp'].str.slice(0, 4)

# 2. Convertiamo in numerico (gestendo eventuali errori con NaN)
triples_df['year'] = pd.to_numeric(triples_df['year'], errors='coerce')

# 3. Filtriamo gli anni ragionevoli (es. dal 1980 in poi) per evitare outlier/errori di OCR
triples_df_filtered = triples_df[triples_df['year'] >= 1980]

# 4. Creazione del grafico della densità delle relazioni
plt.figure(figsize=(12, 6))
sns.histplot(triples_df_filtered['year'], bins=int(triples_df_filtered['year'].max() - triples_df_filtered['year'].min()), kde=True, color='teal')

plt.title('Evoluzione Temporale del Network (RQ1)', fontsize=15)
plt.xlabel('Anno', fontsize=12)
plt.ylabel('Numero di Relazioni Estratte', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

print(f"Relazioni totali analizzabili nel tempo: {len(triples_df_filtered)}")

ModuleNotFoundError: No module named 'seaborn'